# 03 · Data Insights Service

The **Data Insights** service lets a client upload a CSV or Excel file and immediately receive:

1. **KPI summary cards** — rows, columns, completeness %, numeric/categorical columns
2. **Column metadata** — type classification (numeric / categorical / temporal / id / geo)
3. **Correlation matrix** — top Pearson correlations between numeric columns (≥ 0.3 threshold)
4. **AI-generated insight summary** — natural language takeaways from the LLM
5. **Suggested follow-up questions** — 3 questions the user might ask next

After upload, clients can ask natural-language questions about their data in a follow-up conversation.

## Architecture

```
            POST /upload
                │
                ▼
      parse CSV/Excel with pandas
                │
                ▼
      _build_insights()  ─────────────────────────────┐
          │                                           │
          ├─ KPI cards (rows, cols, completeness)     │
          ├─ column classification                    │
          ├─ Pearson correlation matrix               │
          └─ LLM summary + suggested Qs              │
                                                     │
            POST /chat/data  ◄────────────────────────┘
                │
                ▼
      LLM Q&A with full DataFrame context
```

In [ ]:
# %pip install -q pandas langchain-groq python-dotenv openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(dotenv_path=Path('..') / '.env')
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.3)
print('Ready')

Ready


## Create a Sample Dataset

We simulate a marketing dataset with numeric, categorical, temporal, and ID columns.

In [3]:
import io

csv_content = """customer_id,age,income,spend,channel,signup_date,region,churn
1,24,32000,850,organic,2023-01-15,North,0
2,35,58000,2100,paid,2023-02-20,South,0
3,45,75000,3400,organic,2023-01-30,East,0
4,28,41000,1100,referral,2023-03-10,West,1
5,52,92000,4200,paid,2022-12-05,North,0
6,31,49000,1800,organic,2023-04-01,South,0
7,29,38000,900,referral,2023-04-15,East,1
8,41,67000,2700,paid,2023-02-28,West,0
9,23,28000,,organic,2023-05-10,North,1
10,38,81000,3100,paid,2023-03-22,South,0
11,60,105000,5000,referral,2022-11-14,East,0
12,27,35000,600,organic,2023-06-01,West,1
"""

df = pd.read_csv(io.StringIO(csv_content))
print(df.shape)
df.head()

(12, 8)


,customer_id,age,income,spend,channel,signup_date,region,churn
0,1,24,32000,850.0,organic,2023-01-15,North,0
1,2,35,58000,2100.0,paid,2023-02-20,South,0
2,3,45,75000,3400.0,organic,2023-01-30,East,0
3,4,28,41000,1100.0,referral,2023-03-10,West,1
4,5,52,92000,4200.0,paid,2022-12-05,North,0


## Column Classification

In [4]:
def classify_columns(df: pd.DataFrame) -> dict:
    """Classify each column by semantic type."""
    classification = {}
    for col in df.columns:
        col_lower = col.lower()
        series = df[col]

        # ID detection
        if any(kw in col_lower for kw in ('_id', 'id_', 'uuid', 'key')) or \
           (series.dtype in (int, float) and series.nunique() == len(series)):
            classification[col] = 'id'
        # Temporal detection
        elif any(kw in col_lower for kw in ('date', 'time', 'year', 'month', 'week', 'day')):
            classification[col] = 'temporal'
        # Geographic detection
        elif any(kw in col_lower for kw in ('region', 'country', 'city', 'state', 'zip', 'lat', 'lon')):
            classification[col] = 'geo'
        # Numeric
        elif pd.api.types.is_numeric_dtype(series):
            classification[col] = 'numeric'
        # Categorical
        else:
            classification[col] = 'categorical'

    return classification

col_types = classify_columns(df)
for col, kind in col_types.items():
    print(f'  {col:20s} → {kind}')

  customer_id          → id
  age                  → id
  income               → id
  spend                → numeric
  channel              → categorical
  signup_date          → temporal
  region               → geo
  churn                → numeric


## KPI Cards

In [5]:
def compute_kpis(df: pd.DataFrame) -> dict:
    total_cells = df.size
    null_cells = df.isnull().sum().sum()
    completeness = round((1 - null_cells / total_cells) * 100, 1)

    ct = classify_columns(df)
    num_cols  = [c for c, t in ct.items() if t == 'numeric']
    cat_cols  = [c for c, t in ct.items() if t == 'categorical']
    date_cols = [c for c, t in ct.items() if t == 'temporal']

    return {
        'rows': len(df),
        'columns': len(df.columns),
        'completeness_pct': completeness,
        'numeric_cols': len(num_cols),
        'categorical_cols': len(cat_cols),
        'date_cols': len(date_cols),
    }

kpis = compute_kpis(df)
print(json.dumps(kpis, indent=2))

{
  "rows": 12,
  "columns": 8,
  "completeness_pct": 99.0,
  "numeric_cols": 2,
  "categorical_cols": 1,
  "date_cols": 1
}


## Correlation Matrix

In [6]:
def compute_correlations(df: pd.DataFrame, threshold: float = 0.3) -> list:
    """Return top Pearson correlations above the threshold."""
    num_cols = df.select_dtypes(include='number').columns.tolist()
    if len(num_cols) < 2:
        return []

    corr_matrix = df[num_cols].corr(method='pearson')
    pairs = []
    seen = set()

    for i, col1 in enumerate(corr_matrix.columns):
        for j, col2 in enumerate(corr_matrix.columns):
            if i >= j:
                continue
            r = corr_matrix.loc[col1, col2]
            if abs(r) >= threshold and not pd.isna(r):
                key = tuple(sorted([col1, col2]))
                if key not in seen:
                    seen.add(key)
                    pairs.append({'col1': col1, 'col2': col2, 'r': round(float(r), 3)})

    # Sort by absolute correlation descending
    pairs.sort(key=lambda x: abs(x['r']), reverse=True)
    return pairs[:10]

correlations = compute_correlations(df)
print('Strong correlations found:')
for pair in correlations:
    direction = 'positive' if pair['r'] > 0 else 'negative'
    print(f'  {pair["col1"]} ↔ {pair["col2"]}: r={pair["r"]} ({direction})')

Strong correlations found:
  income ↔ spend: r=0.991 (positive)
  age ↔ spend: r=0.977 (positive)
  age ↔ income: r=0.97 (positive)
  income ↔ churn: r=-0.664 (negative)
  spend ↔ churn: r=-0.645 (negative)
  age ↔ churn: r=-0.595 (negative)
  customer_id ↔ churn: r=0.307 (positive)


## LLM-Powered Summary

In [7]:
def build_dataframe_context(df: pd.DataFrame) -> str:
    """Compact representation of the DataFrame for the LLM."""
    lines = [
        f'Shape: {df.shape[0]} rows × {df.shape[1]} columns',
        f'Columns: {list(df.columns)}',
        f'Dtypes:\n{df.dtypes.to_string()}',
        f'\nDescriptive stats:\n{df.describe(include="all").to_string()}',
    ]
    null_counts = df.isnull().sum()
    if null_counts.any():
        lines.append(f'\nMissing values:\n{null_counts[null_counts > 0].to_string()}')

    lines.append(f'\nFirst 5 rows:\n{df.head(5).to_string()}')
    return '\n'.join(lines)


async def summarize_dataframe(df: pd.DataFrame) -> dict:
    ctx = build_dataframe_context(df)
    prompt = (
        'You are a senior data analyst. Based on the dataset below, '
        'provide:\n1. A concise narrative summary (3–5 sentences).\n'
        '2. Three specific follow-up questions the analyst should explore.\n\n'
        f'Dataset:\n{ctx}\n\n'
        'Respond with JSON: {"summary": "...", "questions": ["...", "...", "..."]}'
    )
    msg = await llm.ainvoke([HumanMessage(content=prompt)])
    try:
        raw = msg.content.strip()
        # Strip markdown code fences if present
        if raw.startswith('```'):
            raw = raw.split('```')[1]
            if raw.startswith('json'):
                raw = raw[4:]
        return json.loads(raw.strip())
    except Exception:
        return {'summary': msg.content, 'questions': []}


summary_result = await summarize_dataframe(df)
print('Summary:', summary_result['summary'])
print('\nSuggested follow-ups:')
for q in summary_result.get('questions', []):
    print(f'  • {q}')

Summary: The dataset contains information about 12 customers, including their age, income, spending habits, and whether they have churned. The average customer is around 36 years old, has an income of approximately $58,417, and spends around $2,341. The churn rate is about 33%, indicating that roughly one-third of customers have stopped using the service. There is one missing value in the spend column. The data suggests that customers from different regions and channels have varying spending habits and churn rates.

Suggested follow-ups:
  • What is the relationship between customer age and churn rate, and are there any specific age groups that are more likely to churn?
  • Is there a significant difference in spending habits between customers who signed up through different channels, and does this impact their likelihood of churning?
  • Are there any regional trends in customer spending and churn rates, and can these insights be used to inform targeted marketing strategies?


## Follow-Up Q&A

In [8]:
async def data_qa(question: str, df: pd.DataFrame) -> str:
    ctx = build_dataframe_context(df)
    messages = [
        SystemMessage(
            content='You are a data analyst. Answer the question based only on the dataset provided.'
        ),
        HumanMessage(
            content=f'Dataset:\n{ctx}\n\nQuestion: {question}'
        ),
    ]
    response = await llm.ainvoke(messages)
    return response.content


answer = await data_qa('Which channel has the highest average spend?', df)
print(answer)

To find the channel with the highest average spend, we need to calculate the average spend for each channel. 

Based on the provided dataset, we can calculate the average spend for each channel as follows:

- organic: (850 + 3400) / 2 = 2125
- paid: (2100 + 4200) / 2 = 3150
- referral: 1100

Since there is one missing value in the 'spend' column, we cannot calculate the exact average spend for each channel. However, based on the available data, the 'paid' channel has the highest average spend of 3150. 

Please note that this calculation is based on a limited number of data points and may not be representative of the entire dataset. If the missing value is filled, the calculation may change.


In [9]:
answer2 = await data_qa('What is the churn rate by region?', df)
print(answer2)

To calculate the churn rate by region, we need to calculate the number of customers who have churned (churn = 1) in each region and divide it by the total number of customers in that region.

Based on the provided dataset, we can calculate the churn rate by region as follows:

1. North: There are 3 customers in the North region (customer_id 1, 5, and possibly others not shown in the first 5 rows). Out of these, 0 customers have churned (based on the first 5 rows). However, we cannot determine the exact churn rate for the North region with the given data.

2. South: There is at least 1 customer in the South region (customer_id 2). This customer has not churned. 

3. East: There is at least 1 customer in the East region (customer_id 3). This customer has not churned.

4. West: There is at least 1 customer in the West region (customer_id 4). This customer has churned.

To give an exact answer, we would need to analyze the entire dataset, not just the first 5 rows. However, based on the de

## Summary

- Column classification uses **name heuristics + dtype checks** — no ML required.
- Correlation threshold of **0.3** filters noise while surfacing meaningful relationships.
- The LLM sees a **compact text summary** of the DataFrame (not raw cell values), reducing token count.
- Missing values are highlighted explicitly so the LLM can comment on data quality.